In [ ]:
# Cell 1 — 공통 실행 환경 설정
# 이 Cell은 모든 tutorial Notebook의 첫 번째 Cell이며 여러 번 실행해도 안전합니다.

from pathlib import Path
import importlib.util
import os
import subprocess
import sys

# Google Drive의 project folder가 다르면 이 값만 수정합니다.
COLAB_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/DiffDodree"
)

# 필요하면 환경변수로 project root를 직접 지정할 수 있습니다.
PROJECT_ROOT_OVERRIDE = os.environ.get("GUGAKGEN_PROJECT_ROOT")


def find_local_project_root(start: Path) -> Path:
    """현재 위치에서 pyproject.toml을 기준으로 project root를 찾습니다."""
    candidates = [start.resolve(), *start.resolve().parents]

    for candidate in candidates:
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "project root를 찾지 못했습니다. "
        "GUGAKGEN_PROJECT_ROOT를 지정하거나 project folder에서 Notebook을 실행하세요."
    )


try:
    # 순수 local Python에는 상위 ``google`` namespace가 없을 수 있습니다.
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    # 이미 mount되어 있으면 기존 mount를 그대로 사용합니다.
    drive.mount("/content/drive", force_remount=False)

if PROJECT_ROOT_OVERRIDE:
    project_root = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
elif IN_COLAB:
    project_root = COLAB_PROJECT_ROOT.resolve()
else:
    project_root = find_local_project_root(Path.cwd())

if not project_root.exists():
    raise FileNotFoundError(
        f"project folder가 존재하지 않습니다: {project_root}\n"
        "COLAB_PROJECT_ROOT 또는 GUGAKGEN_PROJECT_ROOT를 확인하세요."
    )

bootstrap_script = project_root / "scripts" / "bootstrap_notebook.py"

if not bootstrap_script.exists():
    raise FileNotFoundError(
        f"Notebook bootstrap script를 찾지 못했습니다: {bootstrap_script}"
    )

environment_profile = "colab" if IN_COLAB else "local"

# 필요한 dependency만 확인하고 누락된 package만 설치합니다.
# Colab profile에서는 Colab이 제공한 CUDA용 PyTorch를 교체하지 않습니다.
subprocess.run(
    [
        sys.executable,
        str(bootstrap_script),
        "--project-root",
        str(project_root),
        "--profile",
        environment_profile,
        "--install-missing",
    ],
    check=True,
)

# 현재 kernel에서 src package를 즉시 import할 수 있게 합니다.
src_root = project_root / "src"

if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

os.environ["GUGAKGEN_PROJECT_ROOT"] = str(project_root)
os.chdir(project_root)

from gugakgen.device import resolve_device
from gugakgen.reproducibility import set_global_seed

runtime = resolve_device(preference="auto")
device = runtime.device

SEED = 2026
set_global_seed(SEED)

print("환경 설정 완료")
print(f"- 실행 환경: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"- Project root: {project_root}")
print(f"- Device: {device}")
print(f"- AMP enabled: {runtime.amp_enabled}")
print(f"- Seed: {SEED}")

# 06. Evaluation과 research artifact export

이 Notebook은 생성 결과를 raw, projected, accepted 단계가 섞이지 않도록 평가합니다.
transition validity, Graph path edit, exact-copy novelty, 환입 satisfaction, Exact-meter validity를
계산하고 paper-ready CSV와 provenance manifest를 저장합니다.

평가 범위는 symbolic 악보입니다. 시김새, 농현, 음색, 연주 timing은 metric과 expert protocol에서
제외하며 향후 연구 범위로 명시합니다.

In [ ]:
import hashlib
import json
from dataclasses import replace

import numpy as np
import pandas as pd
from IPython.display import FileLink, display

from gugakgen.data.musicxml import tokens_to_score, write_musicxml
from gugakgen.workflows.generate import load_generation_bundle

checkpoint_candidates = [
    project_root / "artifacts/checkpoints/yangcheongdodeuri_maskgit.pt",
    project_root / "artifacts/m2_denoiser_v0.pt",
    project_root / "artifacts/checkpoints/yangcheongdodeuri_maskgit_smoke.pt",
]
checkpoint_path = next((path for path in checkpoint_candidates if path.exists()), None)
if checkpoint_path is None:
    raise FileNotFoundError("사용 가능한 Checkpoint가 없습니다.")

bundle = load_generation_bundle(checkpoint_path, project_root=project_root, runtime=runtime)
N_SAMPLES = 4  # tutorial용. 논문 보고에서는 사전등록한 sample 수를 사용합니다.
SEED_START = 4200
EXPORT_COUNT = 2
evaluation_config = replace(
    bundle.sampling_config,
    num_steps=4,
    copy_rejection_attempts=2,
)

print("Checkpoint:", checkpoint_path.relative_to(project_root))
print("SHA-256:", bundle.checkpoint.sha256)
if "smoke" in checkpoint_path.name:
    print("주의: smoke Checkpoint metric은 논문 결과로 사용하지 않습니다.")

In [ ]:
# novelty와 structure metric helper를 정의합니다.
def token_signatures(tokens) -> list[tuple]:
    return [
        (
            tuple(int(value) for value in tokens.pitch[index]),
            tuple(int(value) for value in tokens.duration[index]),
        )
        for index in range(tokens.num_measures)
    ]


original_signatures = token_signatures(bundle.context.tokens)
original_signature_set = set(original_signatures)


def evaluate_sample(seed: int):
    sampled = bundle.sampler.sample(
        seed=seed,
        config=evaluation_config,
        score_anchors=bundle.score_anchors,
        family_anchors=bundle.family_anchors,
    )
    signatures = token_signatures(sampled.tokens)
    exact_copy_any = float(np.mean([value in original_signature_set for value in signatures]))
    exact_copy_position = float(
        np.mean([value == original_signatures[index] for index, value in enumerate(signatures)])
    )
    meter_valid = all(
        bundle.sampler.meter_decoder.is_valid(
            sampled.tokens.pitch[index],
            sampled.tokens.duration[index],
            float(sampled.tokens.target_beats[index]),
        )
        for index in range(sampled.tokens.num_measures)
    )
    metrics = {
        "seed": seed,
        "raw_transition_validity": bundle.context.graph.transition_validity(sampled.raw_family),
        "projected_transition_validity": bundle.context.graph.transition_validity(sampled.tokens.family),
        "structure_edits": sampled.structure_edits,
        "structure_edit_rate": sampled.structure_edits / sampled.tokens.num_measures,
        "exact_copy_any": exact_copy_any,
        "exact_copy_position": exact_copy_position,
        "novel_measure_rate": 1.0 - exact_copy_any,
        "hwanip_family_satisfaction": bundle.context.hwanip_relations.family_satisfaction(
            sampled.tokens.family
        ),
        "hwanip_surface_satisfaction": bundle.context.hwanip_relations.surface_satisfaction(
            signatures
        ),
        "exact_meter_valid": bool(meter_valid),
    }
    return sampled, metrics

In [ ]:
# 여러 Seed의 결과를 생성하고 raw/projected metric을 분리해 수집합니다.
samples = []
metric_rows = []
for offset in range(N_SAMPLES):
    seed = SEED_START + offset
    sampled, metrics = evaluate_sample(seed)
    samples.append(sampled)
    metric_rows.append(metrics)

metrics_table = pd.DataFrame(metric_rows)
display(metrics_table)

metric_columns = [column for column in metrics_table.columns if column != "seed"]
paper_table = pd.DataFrame(
    {
        "metric": metric_columns,
        "mean": [metrics_table[column].astype(float).mean() for column in metric_columns],
        "std": [metrics_table[column].astype(float).std(ddof=1) for column in metric_columns],
        "n": N_SAMPLES,
    }
)
display(paper_table)

In [ ]:
# MusicXML, CSV, JSON manifest를 하나의 evaluation folder에 저장합니다.
output_dir = project_root / "artifacts/evaluation/tutorial_phase1"
output_dir.mkdir(parents=True, exist_ok=True)

metrics_csv = output_dir / "sample_metrics.csv"
paper_csv = output_dir / "paper_summary.csv"
metrics_table.to_csv(metrics_csv, index=False)
paper_table.to_csv(paper_csv, index=False)

exported_files = []
export_config = bundle.context.config["export"]
for sampled, metrics in zip(samples[:EXPORT_COUNT], metric_rows[:EXPORT_COUNT]):
    score = tokens_to_score(
        sampled.tokens,
        bundle.context.vocabulary,
        piece_id=bundle.context.score.piece_id,
        part=bundle.context.score.part,
        meter_id_to_value=bundle.context.meter_id_to_value,
    )
    destination = output_dir / f"yangcheongdodeuri_seed_{metrics['seed']}.musicxml"
    write_musicxml(
        score,
        destination,
        divisions_per_quarter=export_config["divisions_per_quarter"],
        key_fifths=export_config["key_fifths"],
        part_id=export_config["part_id"],
        part_name=export_config["part_name"],
    )
    exported_files.append(destination)


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


manifest_path = output_dir / "summary_manifest.json"
manifest = {
    "schema_version": 1,
    "scope": "symbolic pitch-duration score; 시김새/농현/audio excluded",
    "checkpoint": str(checkpoint_path.relative_to(project_root)),
    "checkpoint_sha256": bundle.checkpoint.sha256,
    "device": bundle.runtime.as_dict(),
    "sample_count": N_SAMPLES,
    "seeds": metrics_table["seed"].tolist(),
    "sampling_config": {
        key: getattr(evaluation_config, key) for key in evaluation_config.__dataclass_fields__
    },
    "metrics": metric_rows,
    "files": [],
}

files_for_manifest = [metrics_csv, paper_csv, *exported_files]
manifest["files"] = [
    {"path": str(path.relative_to(project_root)), "sha256": sha256(path)}
    for path in files_for_manifest
]
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

print("Evaluation artifact:", output_dir.relative_to(project_root))
display(FileLink(str(metrics_csv)), FileLink(str(paper_csv)), FileLink(str(manifest_path)))

## 논문 보고 전 Checklist

- smoke Checkpoint 결과를 제외합니다.
- 모든 condition에 동일한 Seed set과 Sampling 수를 사용합니다.
- raw, guided, projected, accepted metric을 한 값으로 합치지 않습니다.
- mean과 std뿐 아니라 sample-level CSV를 보존합니다.
- 공동연구자의 expert evaluation에서는 pitch, duration, 마디 구성, transition structure,
  환입 관계만 질문하고 시김새·농현·음색은 평가 대상에서 제외합니다.